In [1]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_21001/3003301750.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_atlas = pd.read_csv(


In [3]:
# === Global Configuration and Constants ===
start_sqrt_s = 1  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

sigma_tot_lst = []
sqrt_s_lst = []
error_lst = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0729

model_params = {
    'atlas': {
        'pl':  {'mg': 0.412, 'a1': 1.652, 'a2': 1.479}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}


In [4]:

# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)

    factor = q2 + 9 * abs(k ** 2 - q2 / 4)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func):
    k = sqrt_s * x
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s

    return k * (T_1(k, 0.0, phi, mg, a1, a2, m2_func) - T_2(k, 0.0, phi, mg, a1, a2, m2_func)) * jacobian

def amp_calculation(diff_T, s, epsilon):
    alpha_pomeron = 1.0 + epsilon
    regge_factor = (s / s0) ** alpha_pomeron
    
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


In [5]:
amp_born_lst = []

sqrt_s_lst = []

# === Main Function ===
def main():
    global start_sqrt_s
    global sqrt_s

    max_sqrt_s = 13000
    step = 100
    n_points = 10000

    # Using only PL model with ATLAS
    mass_model = 'pl'
    ensemble = 'atlas'

    fig = go.Figure()

    sigma_tot_lst = []
    

    

    m2_func = get_m2_function(mass_model)
    params = model_params[ensemble][mass_model]
    mg, a1, a2 = params['mg'], params['a1'], params['a2']
    epsilon = epsilon_values[ensemble]

    sqrt_s = start_sqrt_s
    while sqrt_s <= max_sqrt_s:
        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func),
                0, 1,
                n=n_points
            )[0]

        integral_value = fixed_quad(
            inner_integral,
            0, 1,
            n=n_points
        )[0]

        diff_T = integral_value
        s = sqrt_s * sqrt_s

        amp_value = amp_calculation(diff_T, s, epsilon)
        sigma_tot_value = sigma_tot(amp_value, s)

        sigma_tot_lst.append(sigma_tot_value)
        sqrt_s_lst.append(sqrt_s)
        amp_born_lst.append(amp_value)

        sqrt_s += step

    # Add PL model trace
    fig.add_trace(go.Scatter(
        x=sqrt_s_lst,
        y=sigma_tot_lst,
        mode='lines+markers',
        line=dict(
            color='blue',
            width=2
        ),
        marker=dict(
            size=4
        ),
        name='PL Model (ATLAS)'
    ))

    # Add ATLAS data
    fig.add_trace(go.Scatter(
        x=x_atlas,
        y=y_atlas,
        mode='markers',
        marker=dict(
            color='black',
            size=6,
            symbol='square'
        ),
        error_y=dict(
            type='data',
            array=y_error_atlas,
            visible=True
        ),
        name='ATLAS Data'
    ))

    # Configure layout
    fig.update_layout(
        title='Sigma Tot vs. sqrt(s) - PL Model with ATLAS Data',
        xaxis=dict(
            title='sqrt(s) [GeV]',
            type='log',
        ),
        yaxis=dict(
            title='Sigma Tot [mb]',
        ),
        showlegend=True,
        legend=dict(
            title='Model/Data'
        ),
        plot_bgcolor='white',
        hovermode='x unified'
    )
    
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

    # fig.show(renderer="browser")
    # fig.write_html("results/sigma_tot/sigma_tot_pl_atlas.html")
    # fig.write_image("results/sigma_tot/sigma_tot_pl_atlas.pdf", width=1200, height=600)


In [6]:
if __name__ == "__main__":
    main()

In [7]:
lst_s = []
for key, value in enumerate(sqrt_s_lst):
    lst_s.append(value ** 2)
    # print(f"sqrt(s) = {value:.2f} GeV, s = {lst_s[key]:.2f} GeV^2")

# print(amp_born_lst)

def sigma_tot_eik(s, amp):
    return (4*np.pi)/s * amp.imag * 0.389379323


In [8]:
# limit = 1000
# epsrel = 1e-20
# epsabs = 1e-20

# q_upper_limit = 0.2
# b_upper_limit = 10


# def inner_integral(b, s, amp):
#     integrand = lambda q: q * j0(b*q)
#     result, _ = quad(integrand, 0, q_upper_limit, limit=limit, epsabs=epsabs, epsrel=epsrel)  # limite de subdivisões
#     return result * (1/s) * amp


# def outer_integrand(b, s, amp):
#     inner_result = inner_integral(b, s, amp)
#     return 1j * s * b * (1 - np.exp(1j*inner_result))


# def outer_real(b, s, amp):
#     return np.real(outer_integrand(b, s, amp))

# def outer_imag(b, s, amp):
#     return np.imag(outer_integrand(b, s, amp))



# def compute_double_integral(s, amp):
#     real_part, _ = quad(lambda b: outer_real(b, s, amp), 0, b_upper_limit, limit=limit, epsabs=epsabs, epsrel=epsrel)
#     imag_part, _ = quad(lambda b: outer_imag(b, s, amp), 0, b_upper_limit, limit=limit, epsabs=epsabs, epsrel=epsrel)
#     return real_part + 1j * imag_part


# lst_amp_eik = []

# for s_val, amp_born_val in zip(lst_s, amp_born_lst):
#     amp_eik_val = compute_double_integral(s_val, amp_born_val)
#     # print(f"Numerical result: {result1} for s = {s_val} and amp born = {amp_born_val}")
#     lst_amp_eik.append(amp_eik_val)

# print(lst_amp_eik)


In [9]:
from scipy.integrate import quad
import numpy as np
from scipy.special import j0

# Limits
q_upper = 0.1  # inner integral
b_upper = 30 # outer integral

# Inner integral over q
def inner_integral(b, s, amp):
    integrand = lambda q: q * j0(b * q)  # you can later include s, amp if needed
    result, _ = quad(integrand, 0, q_upper, limit=200)
    return result * (1/s) * amp

# Outer integrand
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j*inner_result)) # amp applied here as example

# Split real and imaginary parts
def outer_real(b, s, amp):
    return np.real(outer_integrand(b, s, amp))

def outer_imag(b, s, amp):
    return np.imag(outer_integrand(b, s, amp))

# Compute double integral for given s, amp
def compute_double_integral(s, amp):
    real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp), limit=500)
    imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp), limit=500)
    return real_part + 1j * imag_part

lst_amp_eik = []

# Example loop
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    # print(f"s={s}, amp={amp:.2e}, result={amp_eik_val:.2e}")
    lst_amp_eik.append(amp_eik_val)

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

print(f'{lst_amp_eik[-1].imag:.3e}')
print(f'{lst_sigma_tot_eik[-1]:.2f}')

3.797e+10
1116.15


In [10]:
import numpy as np
from scipy.integrate import quad, simpson
from scipy.special import j0

# Limits
q_upper = 0.1  # inner integral
b_upper = 30   # outer integral

# Number of points for Simpson's rule
n_q = 200  # must be odd for simpson, but scipy handles even too

# Inner integral using Simpson's rule
def inner_integral(b, s, amp):
    q_vals = np.linspace(0, q_upper, n_q)
    integrand_vals = q_vals * j0(b * q_vals)  # you can include s, amp if needed
    result = simpson(integrand_vals, q_vals)
    return result * (1/s) * amp

# Outer integrand
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j * inner_result))

# Split real and imaginary parts
def outer_real(b, s, amp):
    return np.real(outer_integrand(b, s, amp))

def outer_imag(b, s, amp):
    return np.imag(outer_integrand(b, s, amp))

# Compute double integral for given s, amp
def compute_double_integral(s, amp):
    real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp), limit=500)
    imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp), limit=500)
    return real_part + 1j * imag_part

# Example usage
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    lst_amp_eik.append(amp_eik_val)

lst_sigma_tot_eik = []
for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

print(f'{lst_amp_eik[-1].imag:.3e}')
print(f'{lst_sigma_tot_eik[-1]:.2f}')


3.797e+10
1116.15


In [11]:
import numpy as np
from scipy.integrate import quad, simpson
from scipy.special import j0

# Limits
q_upper = 0.1  # inner integral
b_upper = 30   # outer integral

# Number of points for Simpson's rule (outer integral)
n_b = 2001  # should be odd for Simpson

# Inner integral using quad
def inner_integral(b, s, amp):
    integrand = lambda q: q * j0(b * q)
    result, _ = quad(integrand, 0, q_upper, limit=200)
    return result * (1/s) * amp

# Outer integrand (scalar b)
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j * inner_result))

# Compute double integral for given s, amp (outer by Simpson)
def compute_double_integral(s, amp):
    b_vals = np.linspace(0, b_upper, n_b)
    integrand_vals = [outer_integrand(b, s, amp) for b in b_vals]  # loop over b
    integrand_vals = np.array(integrand_vals)
    result = simpson(integrand_vals, b_vals)
    return result

# Example usage
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    lst_amp_eik.append(amp_eik_val)

lst_sigma_tot_eik = []
for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

print(f'{lst_amp_eik[-1].imag:.3e}')
print(f'{lst_sigma_tot_eik[-1]:.2f}')


3.797e+10
1116.15


In [12]:
from scipy.integrate import quad, fixed_quad, simpson, romb
import numpy as np
from scipy.special import j0

# Limits
q_upper = 0.1  # inner integral
b_upper = 30   # outer integral

# Inner integral
def inner_integral(b, s, amp, method="quad", N=200):
    if method == "quad":
        integrand = lambda q: q * j0(b * q)
        result, _ = quad(integrand, 0, q_upper, limit=200)
    else:
        q_vals = np.linspace(0, q_upper, N+1)
        f_vals = q_vals * j0(b * q_vals)
        if method == "simpson":
            result = simpson(f_vals, dx=q_upper/N)
        elif method == "romb":
            # Romberg requires N = 2^k
            if not ((N & (N-1)) == 0) or N == 0:
                raise ValueError("N must be a power of 2 for Romberg integration")
            result = romb(f_vals, dx=q_upper/N)
        elif method == "fixed":
            result, _ = fixed_quad(lambda q: q*j0(b*q), 0, q_upper, n=N)
        elif method == "mc":
            q_samples = np.random.rand(N) * q_upper
            f_samples = q_samples * j0(b*q_samples)
            result = q_upper * np.mean(f_samples)
        else:
            raise ValueError(f"Unknown method: {method}")
    return result * (1/s) * amp

# Outer integrand
def outer_integrand(b, s, amp, method="quad", N=200):
    inner_result = inner_integral(b, s, amp, method=method, N=N)
    return 1j * s * b * (1 - np.exp(1j*inner_result))

def outer_real(b, s, amp, method="quad", N=200):
    return np.real(outer_integrand(b, s, amp, method=method, N=N))

def outer_imag(b, s, amp, method="quad", N=200):
    return np.imag(outer_integrand(b, s, amp, method=method, N=N))

# Compute double integral
def compute_double_integral(s, amp, method="quad", N=200):
    if method == "quad":
        real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp, method, N), limit=500)
        imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp, method, N), limit=500)
    else:
        b_vals = np.linspace(0, b_upper, N+1)
        real_vals = [outer_real(b, s, amp, method, N) for b in b_vals]
        imag_vals = [outer_imag(b, s, amp, method, N) for b in b_vals]
        if method in ["simpson", "romb"]:
            real_part = simpson(real_vals, dx=b_upper/N) if method=="simpson" else romb(real_vals, dx=b_upper/N)
            imag_part = simpson(imag_vals, dx=b_upper/N) if method=="simpson" else romb(imag_vals, dx=b_upper/N)
        elif method in ["fixed", "mc"]:
            real_part = simpson(real_vals, dx=b_upper/N)
            imag_part = simpson(imag_vals, dx=b_upper/N)
        else:
            raise ValueError(f"Unknown method: {method}")
    return real_part + 1j * imag_part


In [13]:
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp, method="simpson", N=500)
    lst_amp_eik.append(amp_eik_val)
print(f'{lst_amp_eik[-1]}')

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)
print(f'{lst_sigma_tot_eik[-1]:.2f}')


37965332219.913216j
1116.15


In [14]:
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp, method="quad", N=500)
    lst_amp_eik.append(amp_eik_val)
print(f'{lst_amp_eik[-1]}')
lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)
print(f'{lst_sigma_tot_eik[-1]:.2f}')


37965332219.69779j
1116.15


In [15]:
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp, method="fixed", N=500)
    lst_amp_eik.append(amp_eik_val)
print(f'{lst_amp_eik[-1]}')

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)
print(f'{lst_sigma_tot_eik[-1]:.2f}')



37965332219.779884j
1116.15


In [16]:
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp, method="romb", N=1024)
    lst_amp_eik.append(amp_eik_val)
print(f'{lst_amp_eik[-1]}')

lst_sigma_tot_eik = []

for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)

    # print(f's = {s_val}, amp eik = {amp_eik_val}, sigma tot eik = {sigma_tot_eik_val}')
    
    lst_sigma_tot_eik.append(sigma_tot_eik_val)
print(f'{lst_sigma_tot_eik[-1]:.2f}')



37965332219.6978j
1116.15


In [17]:
from scipy.integrate import quad
import numpy as np
from scipy.special import j1  # Bessel J1

# Limits
q_upper = 0.1   # inner integral limit
b_upper = 30    # outer integral limit

# Inner integral over q (analytical)
def inner_integral(b, s, amp):
    # Avoid division by zero at b=0
    if b == 0:
        result = 0.5 * q_upper**2   # limit b→0 gives ∫0^qmax q dq = qmax^2/2
    else:
        result = (q_upper / b) * j1(b * q_upper)
    return result * (1/s) * amp

# Outer integrand
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - np.exp(1j * inner_result))

# Split real and imaginary parts
def outer_real(b, s, amp):
    return np.real(outer_integrand(b, s, amp))

def outer_imag(b, s, amp):
    return np.imag(outer_integrand(b, s, amp))

# Compute double integral for given s, amp
def compute_double_integral(s, amp):
    real_part, _ = quad(outer_real, 0, b_upper, args=(s, amp), limit=500)
    imag_part, _ = quad(outer_imag, 0, b_upper, args=(s, amp), limit=500)
    return real_part + 1j * imag_part

# Example usage with your lists
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    lst_amp_eik.append(amp_eik_val)

lst_sigma_tot_eik = []
for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
    lst_sigma_tot_eik.append(sigma_tot_eik_val)

print(f'{lst_amp_eik[-1].imag:.3e}')
print(f'{lst_sigma_tot_eik[-1]:.2f}')


3.797e+10
1116.15


In [18]:
import numpy as np   # keep numpy only for arrays/lists if needed
from mpmath import quad, besselj, exp, mp

# set working precision (decimal places)
mp.dps = 50  

# Limits
q_upper = 0.1   # inner integral
b_upper = 30    # outer integral

# Inner integral over q (mpmath)
def inner_integral(b, s, amp):
    integrand = lambda q: q * besselj(0, b * q)  # J0 from mpmath
    result = quad(integrand, [0, q_upper])
    return result * (1/s) * amp

# Outer integrand (mpmath)
def outer_integrand(b, s, amp):
    inner_result = inner_integral(b, s, amp)
    return 1j * s * b * (1 - exp(1j * inner_result))  # exp from mpmath

# Compute double integral for given s, amp
def compute_double_integral(s, amp):
    result = quad(lambda b: outer_integrand(b, s, amp), [0, b_upper])
    return result  # already complex (mpmath.mpc)

# Example usage (assuming you already defined lst_s, amp_born_lst, sigma_tot_eik)
lst_amp_eik = []
for s, amp in zip(lst_s, amp_born_lst):
    amp_eik_val = compute_double_integral(s, amp)
    lst_amp_eik.append(amp_eik_val)

lst_sigma_tot_eik = []
for amp_eik_val, s_val in zip(lst_amp_eik, lst_s):
    sigma_tot_eik_val = sigma_tot_eik(s_val, amp_eik_val)
    lst_sigma_tot_eik.append(sigma_tot_eik_val)
